# Dataset & Formulasi Masalah - Amazon Delivery Dataset

Notebook ini dibuat untuk bagian **Peran 2: Dataset & Formulasi Masalah** pada proyek akhir Algoritma Evolusi dan Kecerdasan Berkoloni.

Dataset yang digunakan: [Amazon Delivery Dataset](https://www.kaggle.com/datasets/sujalsuthar/amazon-delivery-dataset)

Tujuan notebook ini:
1. Mengunduh dataset dari Kaggle.
2. Menyimpan data mentah ke folder `data/raw`.
3. Mengecek struktur, missing value, dan tipe data.
4. Membersihkan data agar siap dipakai untuk optimasi.
5. Menyimpan data bersih ke folder `data/processed`.
6. Menjelaskan alasan setiap keputusan preprocessing.

Catatan: jika dijalankan di komputer lokal, pastikan internet aktif. Untuk dataset publik, `kagglehub` biasanya dapat langsung mengunduh dataset. Jika gagal karena autentikasi, login Kaggle atau jalankan notebook ini di Kaggle Notebook.

## 1. Import Library dan Menyiapkan Folder

Kita membuat dua folder utama:
- `data/raw`: menyimpan file asli dari Kaggle tanpa diubah.
- `data/processed`: menyimpan file hasil pembersihan.

Pemisahan ini penting agar data mentah tetap aman dan proses preprocessing dapat diulang jika diperlukan.

In [97]:
from pathlib import Path
import importlib.util
import shutil
import subprocess
import sys
import time
import warnings

PROJECT_ROOT = Path.cwd()
REQUIREMENTS_FILE = PROJECT_ROOT / "requirements.txt"

PACKAGE_IMPORT_MAP = {
    "numpy": "numpy",
    "pandas": "pandas",
    "kagglehub": "kagglehub",
}


def ensure_package(package_name):
    import_name = PACKAGE_IMPORT_MAP.get(package_name, package_name.replace("-", "_"))
    if importlib.util.find_spec(import_name) is not None:
        print(f"Package '{package_name}' sudah tersedia.")
        return
    print(f"Package '{package_name}' belum ada. Mencoba install...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])


def install_missing_requirements(requirements_path):
    if not requirements_path.exists():
        print("requirements.txt tidak ditemukan. Menggunakan package minimum.")
        packages = ["numpy", "pandas", "kagglehub"]
    else:
        packages = []
        for line in requirements_path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            packages.append(line)

    for package in packages:
        ensure_package(package)


install_missing_requirements(REQUIREMENTS_FILE)

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 80)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DATASET_SLUG = "sujalsuthar/amazon-delivery-dataset"

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DIR)
print("Processed data folder:", PROCESSED_DIR)


Package 'pandas' sudah tersedia.
Package 'numpy' sudah tersedia.
Package 'kagglehub' sudah tersedia.
Project root: c:\Users\piosg\Documents\Evolusi
Raw data folder: c:\Users\piosg\Documents\Evolusi\data\raw
Processed data folder: c:\Users\piosg\Documents\Evolusi\data\processed


## 2. Mengunduh Dataset dari Kaggle

Notebook ini memakai `kagglehub` karena lebih mudah untuk mengambil dataset publik dari Kaggle.

Keputusan yang dilakukan:
- File yang diunduh dari Kaggle disalin ke `data/raw`.
- File mentah tidak dimodifikasi supaya bisa dijadikan arsip dan bukti sumber data asli.
- Jika folder `data/raw` sudah berisi file dataset, file tetap dapat ditimpa dengan versi unduhan terbaru.

In [98]:
import kagglehub

download_path = Path(kagglehub.dataset_download(DATASET_SLUG))
print("Dataset berhasil diunduh ke cache KaggleHub:", download_path)

copied_files = []
for source_file in download_path.rglob("*"):
    if source_file.is_file():
        target_file = RAW_DIR / source_file.name
        shutil.copy2(source_file, target_file)
        copied_files.append(target_file)

print("File yang tersimpan di data/raw:")
for file in copied_files:
    print("-", file.name, f"({file.stat().st_size:,} bytes)")

Dataset berhasil diunduh ke cache KaggleHub: C:\Users\piosg\.cache\kagglehub\datasets\sujalsuthar\amazon-delivery-dataset\versions\1
File yang tersimpan di data/raw:
- amazon_delivery.csv (5,971,175 bytes)


## 3. Mengecek File Mentah

Pada tahap ini kita mencari file CSV di folder `data/raw`, lalu memilih file CSV terbesar sebagai dataset utama. Alasan memilih file terbesar adalah karena dataset utama biasanya berisi tabel data pengiriman, sedangkan file kecil biasanya hanya metadata atau contoh.

In [99]:
raw_files = sorted([file for file in RAW_DIR.rglob("*") if file.is_file() and file.name != ".gitkeep"])
csv_files = sorted(RAW_DIR.rglob("*.csv"), key=lambda file: file.stat().st_size, reverse=True)

print("Daftar file di data/raw:")
for file in raw_files:
    print("-", file.name, f"({file.stat().st_size:,} bytes)")

if not csv_files:
    raise FileNotFoundError("Tidak ada file CSV di data/raw. Cek hasil download Kaggle.")

DATA_FILE = csv_files[0]
print("\nFile CSV utama yang dipakai:", DATA_FILE.name)

Daftar file di data/raw:
- amazon_delivery.csv (5,971,175 bytes)

File CSV utama yang dipakai: amazon_delivery.csv


## 4. Membaca Dataset dan Melihat Kondisi Awal

Sebelum membersihkan data, kita perlu melihat:
- jumlah baris dan kolom,
- nama kolom,
- contoh isi data,
- tipe data,
- jumlah missing value.

Langkah ini penting agar keputusan preprocessing tidak asal dilakukan.

In [100]:
df_raw = pd.read_csv(DATA_FILE)

print("Ukuran data mentah:", df_raw.shape)
display(df_raw.head())

print("\nNama kolom:")
print(list(df_raw.columns))

print("\nTipe data awal:")
display(df_raw.dtypes.to_frame("dtype"))

print("\nMissing value awal:")
missing_raw = df_raw.isna().sum().sort_values(ascending=False).to_frame("missing_count")
missing_raw["missing_percent"] = (missing_raw["missing_count"] / len(df_raw) * 100).round(2)
display(missing_raw)

Ukuran data mentah: (43739, 16)


,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys



Nama kolom:
['Order_ID', 'Agent_Age', 'Agent_Rating', 'Store_Latitude', 'Store_Longitude', 'Drop_Latitude', 'Drop_Longitude', 'Order_Date', 'Order_Time', 'Pickup_Time', 'Weather', 'Traffic', 'Vehicle', 'Area', 'Delivery_Time', 'Category']

Tipe data awal:


,dtype
Order_ID,object
Agent_Age,int64
Agent_Rating,float64
Store_Latitude,float64
Store_Longitude,float64
Drop_Latitude,float64
Drop_Longitude,float64
Order_Date,object
Order_Time,object
Pickup_Time,object



Missing value awal:


,missing_count,missing_percent
Weather,91,0.21
Agent_Rating,54,0.12
Order_ID,0,0.00
Agent_Age,0,0.00
Store_Latitude,0,0.00
Store_Longitude,0,0.00
Drop_Latitude,0,0.00
Drop_Longitude,0,0.00
Order_Date,0,0.00
Order_Time,0,0.00


## 5. Standarisasi Nama Kolom

Nama kolom distandarkan menjadi huruf kecil dan memakai underscore. Contoh: `Delivery Time` atau `Delivery_Time` menjadi `delivery_time`.

Alasannya:
- memudahkan penulisan kode,
- mengurangi risiko error karena spasi atau huruf besar/kecil,
- membuat format kolom konsisten untuk tahap implementasi algoritma.

In [101]:
def standardize_column_name(column_name):
    cleaned = str(column_name).strip().lower()
    cleaned = cleaned.replace(" ", "_").replace("-", "_").replace("/", "_")
    while "__" in cleaned:
        cleaned = cleaned.replace("__", "_")
    return cleaned.strip("_")


df = df_raw.copy()
original_columns = list(df.columns)
df.columns = [standardize_column_name(col) for col in df.columns]

column_mapping = pd.DataFrame({
    "original_column": original_columns,
    "standardized_column": df.columns
})

display(column_mapping)
print("Kolom setelah distandarkan:")
print(list(df.columns))

,original_column,standardized_column
0,Order_ID,order_id
1,Agent_Age,agent_age
2,Agent_Rating,agent_rating
3,Store_Latitude,store_latitude
4,Store_Longitude,store_longitude
5,Drop_Latitude,drop_latitude
6,Drop_Longitude,drop_longitude
7,Order_Date,order_date
8,Order_Time,order_time
9,Pickup_Time,pickup_time


Kolom setelah distandarkan:
['order_id', 'agent_age', 'agent_rating', 'store_latitude', 'store_longitude', 'drop_latitude', 'drop_longitude', 'order_date', 'order_time', 'pickup_time', 'weather', 'traffic', 'vehicle', 'area', 'delivery_time', 'category']


## 6. Membersihkan Nilai Kosong, Duplikasi, dan Tipe Data

Keputusan preprocessing:
- Nilai kosong berbentuk teks seperti `NaN`, `null`, atau string kosong disamakan menjadi `NaN`.
- Baris duplikat dihapus karena dapat membuat hasil analisis bias.
- Kolom numerik penting dikonversi menjadi angka.
- Rating kurir harus berada pada rentang 1 sampai 5. Nilai di luar rentang tersebut dianggap tidak valid dan diisi dengan median.
- Umur kurir yang tidak realistis juga diisi dengan median agar tidak mengganggu analisis.
- Baris tanpa koordinat penting dihapus karena koordinat dibutuhkan untuk menghitung jarak.
- `delivery_time` tetap divalidasi jika tersedia, tetapi tidak digunakan sebagai input optimasi karena pada kasus nyata nilainya belum diketahui sebelum pengiriman selesai.

Untuk kategori seperti cuaca, traffic, kendaraan, dan area, nilai kosong diisi `Unknown` agar data tidak terlalu banyak terbuang.

In [102]:
before_rows = len(df)

df = df.replace(["", " ", "NaN", "nan", "NULL", "null", "None", "none"], np.nan)
df = df.drop_duplicates()

if "order_id" in df.columns:
    df = df.drop_duplicates(subset=["order_id"], keep="first")

numeric_candidates = [
    "agent_age", "agent_rating",
    "store_latitude", "store_longitude",
    "drop_latitude", "drop_longitude",
    "delivery_time"
]

for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

categorical_candidates = ["weather", "traffic", "vehicle", "area", "category"]
if "agent_rating" in df.columns:
    df.loc[~df["agent_rating"].between(1, 5), "agent_rating"] = np.nan
    df["agent_rating"] = df["agent_rating"].fillna(df["agent_rating"].median()).round(2)

if "agent_age" in df.columns:
    df.loc[~df["agent_age"].between(18, 70), "agent_age"] = np.nan
    df["agent_age"] = df["agent_age"].fillna(df["agent_age"].median()).round().astype(int)

for col in categorical_candidates:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()
        df[col] = df[col].replace(["", "Na", "Nan", "NaN", "N/A", "None", "Null", "<NA>"], pd.NA)
        df[col] = df[col].str.title().fillna("Unknown")

mandatory_columns = [
    col for col in [
        "store_latitude", "store_longitude",
        "drop_latitude", "drop_longitude"
    ] if col in df.columns
]

df = df.dropna(subset=mandatory_columns)

if {"store_latitude", "drop_latitude"}.issubset(df.columns):
    df = df[df["store_latitude"].between(-90, 90)]
    df = df[df["drop_latitude"].between(-90, 90)]

if {"store_longitude", "drop_longitude"}.issubset(df.columns):
    df = df[df["store_longitude"].between(-180, 180)]
    df = df[df["drop_longitude"].between(-180, 180)]

if "delivery_time" in df.columns:
    df = df[df["delivery_time"] > 0]

after_rows = len(df)
print("Jumlah baris awal:", before_rows)
print("Jumlah baris setelah cleaning dasar:", after_rows)
print("Jumlah baris terhapus:", before_rows - after_rows)

Jumlah baris awal: 43739
Jumlah baris setelah cleaning dasar: 43739
Jumlah baris terhapus: 0


## 7. Menambahkan Fitur Jarak Pengiriman

Dataset memiliki koordinat toko dan koordinat tujuan. Dari koordinat tersebut, kita menghitung estimasi jarak menggunakan rumus Haversine.

Alasan fitur ini dibuat:
- Jarak adalah salah satu faktor utama yang memengaruhi waktu pengiriman.
- Jarak dapat dipakai dalam fungsi objektif Genetic Algorithm.
- Data menjadi lebih siap digunakan untuk optimasi, bukan hanya analisis deskriptif.

Karena konteks dataset adalah last-mile delivery, jarak di atas 100 km dianggap tidak realistis dan dibuang sebagai outlier koordinat.

In [103]:
def haversine_km(lat1, lon1, lat2, lon2):
    radius_earth_km = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return radius_earth_km * c


required_location_cols = {"store_latitude", "store_longitude", "drop_latitude", "drop_longitude"}

if required_location_cols.issubset(df.columns):
    df["distance_km"] = haversine_km(
        df["store_latitude"],
        df["store_longitude"],
        df["drop_latitude"],
        df["drop_longitude"]
    ).round(3)
    rows_before_distance_filter = len(df)
    df = df[df["distance_km"].between(0, 100)]
    removed_distance_outliers = rows_before_distance_filter - len(df)
    print("Fitur distance_km berhasil dibuat.")
    print("Baris outlier jarak > 100 km yang dihapus:", removed_distance_outliers)
else:
    print("Kolom koordinat belum lengkap, distance_km tidak dibuat.")

display(df[[col for col in ["store_latitude", "store_longitude", "drop_latitude", "drop_longitude", "distance_km"] if col in df.columns]].head())

Fitur distance_km berhasil dibuat.
Baris outlier jarak > 100 km yang dihapus: 188


,store_latitude,store_longitude,drop_latitude,drop_longitude,distance_km
0,22.745049,75.892471,22.765049,75.912471,3.025
1,12.913041,77.683237,13.043041,77.813237,20.184
2,12.914264,77.678400,12.924264,77.688400,1.553
3,11.003669,76.976494,11.053669,77.026494,7.790
4,12.972793,80.249982,13.012793,80.289982,6.210


## 8. Membuat Kolom Penalti untuk Optimasi

Agar data dapat dipakai oleh Genetic Algorithm, beberapa kolom kategori diubah menjadi nilai numerik sederhana.

Contoh alasan:
- Traffic yang lebih padat diberi penalti lebih besar.
- Cuaca buruk diberi penalti lebih besar.
- Jenis kendaraan dapat diberi penalti berbeda karena berpengaruh pada kecepatan pengiriman.
- Rating kurir dapat diberi penalti: rating lebih rendah berarti risiko operasional lebih tinggi.

Nilai penalti ini tidak dimaksudkan sebagai kebenaran mutlak, tetapi sebagai pendekatan awal agar masalah optimasi bisa dimodelkan.

In [104]:
traffic_penalty_map = {
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "Jam": 4,
    "Very High": 4,
    "Unknown": 2
}

weather_penalty_map = {
    "Sunny": 1,
    "Cloudy": 2,
    "Windy": 2,
    "Fog": 3,
    "Sandstorms": 3,
    "Stormy": 4,
    "Rainy": 4,
    "Unknown": 2
}

vehicle_penalty_map = {
    "Bike": 1,
    "Motorcycle": 1,
    "Scooter": 1,
    "Electric Scooter": 1,
    "Van": 2,
    "Car": 2,
    "Bicycle": 3,
    "Unknown": 2
}

if "traffic" in df.columns:
    df["traffic_penalty"] = df["traffic"].map(traffic_penalty_map).fillna(2).astype(int)

if "weather" in df.columns:
    df["weather_penalty"] = df["weather"].map(weather_penalty_map).fillna(2).astype(int)

if "vehicle" in df.columns:
    df["vehicle_penalty"] = df["vehicle"].map(vehicle_penalty_map).fillna(2).astype(int)

if "agent_rating" in df.columns:
    df["agent_penalty"] = (6 - df["agent_rating"]).round(3)

penalty_cols = [col for col in ["traffic", "traffic_penalty", "weather", "weather_penalty", "vehicle", "vehicle_penalty", "agent_rating", "agent_penalty"] if col in df.columns]
display(df[penalty_cols].head())

,traffic,traffic_penalty,weather,weather_penalty,vehicle,vehicle_penalty,agent_rating,agent_penalty
0,High,3,Sunny,1,Motorcycle,1,4.9,1.1
1,Jam,4,Stormy,4,Scooter,1,4.5,1.5
2,Low,1,Sandstorms,3,Motorcycle,1,4.4,1.6
3,Medium,2,Sunny,1,Motorcycle,1,4.7,1.3
4,High,3,Cloudy,2,Scooter,1,4.6,1.4


## 9. Membuat Feature Score untuk Optimasi Bobot

Untuk formulasi akhir, Genetic Algorithm tidak langsung mengoptimasi urutan order. Genetic Algorithm digunakan untuk mencari bobot terbaik dari beberapa faktor risiko pengiriman.

Setiap faktor diubah dulu menjadi skor 0 sampai 1:

- `distance_score`
- `traffic_score`
- `weather_score`
- `vehicle_score`
- `agent_score`

Lima skor tersebut menjadi input untuk formula prioritas berbobot:

`priority_score_i = w_distance * distance_score_i + w_traffic * traffic_score_i + w_weather * weather_score_i + w_vehicle * vehicle_score_i + w_agent * agent_score_i`

Pada tahap preprocessing, notebook juga membuat `priority_risk_score` sebagai baseline sederhana dengan bobot sama rata. Nantinya, bagian implementasi algoritma dapat mengganti bobot sama rata tersebut dengan bobot hasil Genetic Algorithm.

`delivery_time` tidak dimasukkan sebagai input skor prioritas karena pada kasus nyata nilai ini belum diketahui sebelum pengiriman selesai. Kolom tersebut hanya digunakan sebagai target evaluasi historis.

In [105]:
def minmax_score(series):
    min_value = series.min()
    max_value = series.max()
    if max_value == min_value:
        return pd.Series(0.0, index=series.index)
    return ((series - min_value) / (max_value - min_value)).round(6)


score_sources = {
    "distance_score": "distance_km",
    "traffic_score": "traffic_penalty",
    "weather_score": "weather_penalty",
    "vehicle_score": "vehicle_penalty",
    "agent_score": "agent_penalty"
}

score_columns = []
for score_col, source_col in score_sources.items():
    if source_col in df.columns:
        df[score_col] = minmax_score(df[source_col])
        score_columns.append(score_col)

raw_cost_components = [col for col in ["distance_km", "traffic_penalty", "weather_penalty", "vehicle_penalty", "agent_penalty"] if col in df.columns]

if raw_cost_components:
    df["optimization_cost"] = df[raw_cost_components].sum(axis=1).round(3)

if score_columns:
    df["priority_risk_score"] = df[score_columns].mean(axis=1).round(6)
    print("Feature score untuk optimasi bobot:", score_columns)
else:
    print("Tidak ada feature score yang tersedia.")

display_cols = raw_cost_components + score_columns + ["optimization_cost", "priority_risk_score"]
display(df[[col for col in display_cols if col in df.columns]].head())

Feature score untuk optimasi bobot: ['distance_score', 'traffic_score', 'weather_score', 'vehicle_score', 'agent_score']


,distance_km,traffic_penalty,weather_penalty,vehicle_penalty,agent_penalty,distance_score,traffic_score,weather_score,vehicle_score,agent_score,optimization_cost,priority_risk_score
0,3.025,3,1,1,1.1,0.079984,0.666667,0.000000,0.0,0.025,9.125,0.154330
1,20.184,4,4,1,1.5,0.959752,1.000000,1.000000,0.0,0.125,30.684,0.616950
2,1.553,1,3,1,1.6,0.004512,0.000000,0.666667,0.0,0.150,8.153,0.164236
3,7.790,2,1,1,1.3,0.324292,0.333333,0.000000,0.0,0.075,13.090,0.146525
4,6.210,3,2,1,1.4,0.243283,0.666667,0.333333,0.0,0.100,13.610,0.268657


## 10. Mengecek Ulang Dataset Setelah Dibersihkan

Setelah preprocessing, kita cek ulang data untuk memastikan:
- ukuran data setelah cleaning,
- missing value berkurang,
- tipe data sudah sesuai,
- fitur baru berhasil dibuat.

Bagian ini bisa dimasukkan ke laporan sebagai bukti bahwa data sudah dipersiapkan dengan benar.

In [106]:
print("Ukuran data mentah:", df_raw.shape)
print("Ukuran data bersih:", df.shape)

missing_clean = df.isna().sum().sort_values(ascending=False).to_frame("missing_count")
missing_clean["missing_percent"] = (missing_clean["missing_count"] / len(df) * 100).round(2)
display(missing_clean.head(20))

print("\nTipe data setelah preprocessing:")
display(df.dtypes.to_frame("dtype"))

print("\nStatistik numerik:")
display(df.describe(include=[np.number]).T)

Ukuran data mentah: (43739, 16)
Ukuran data bersih: (43551, 28)


,missing_count,missing_percent
order_id,0,0.0
agent_age,0,0.0
optimization_cost,0,0.0
agent_score,0,0.0
vehicle_score,0,0.0
weather_score,0,0.0
traffic_score,0,0.0
distance_score,0,0.0
agent_penalty,0,0.0
vehicle_penalty,0,0.0



Tipe data setelah preprocessing:


,dtype
order_id,object
agent_age,int32
agent_rating,float64
store_latitude,float64
store_longitude,float64
drop_latitude,float64
drop_longitude,float64
order_date,object
order_time,object
pickup_time,object



Statistik numerik:


,count,mean,std,min,25%,50%,75%,max
agent_age,43551.0,29.569631,5.784310,20.000,25.000000,30.000000,35.000000,50.000000
agent_rating,43551.0,4.633088,0.326741,1.000,4.500000,4.700000,4.900000,5.000000
store_latitude,43551.0,17.377891,7.341276,0.000,12.934179,18.554382,22.732225,30.914057
store_longitude,43551.0,70.723177,21.189609,0.000,73.170283,75.898497,78.044095,88.433452
drop_latitude,43551.0,17.441493,7.342636,0.010,12.985519,18.632450,22.783839,31.054057
drop_longitude,43551.0,70.786779,21.189788,0.010,73.277753,75.999490,78.099641,88.563452
delivery_time,43551.0,124.929783,51.939924,10.000,90.000000,125.000000,160.000000,270.000000
distance_km,43551.0,9.733996,5.604431,1.465,4.663000,9.220000,13.681000,20.969000
traffic_penalty,43551.0,2.383344,1.245253,1.000,1.000000,2.000000,4.000000,4.000000
weather_penalty,43551.0,2.511722,0.954556,1.000,2.000000,3.000000,3.000000,4.000000


## 11. Ringkasan Kolom Kategori

Kolom kategori dicek untuk melihat variasi nilai seperti cuaca, traffic, kendaraan, area, dan kategori produk. Informasi ini berguna untuk menjelaskan karakteristik dataset dalam laporan.

In [107]:
for col in ["weather", "traffic", "vehicle", "area", "category"]:
    if col in df.columns:
        print(f"\nDistribusi kolom {col}:")
        display(df[col].value_counts(dropna=False).to_frame("count"))


Distribusi kolom weather:


,count
weather,
Fog,7415
Stormy,7352
Cloudy,7264
Sandstorms,7215
Windy,7198
Sunny,7048
Unknown,59



Distribusi kolom traffic:


,count
traffic,
Low,14940
Jam,13678
Medium,10595
High,4279
Unknown,59



Distribusi kolom vehicle:


,count
vehicle,
Motorcycle,25428
Scooter,14582
Van,3530
Bicycle,11



Distribusi kolom area:


,count
area,
Metropolitian,32565
Urban,9701
Other,1133
Semi-Urban,152



Distribusi kolom category:


,count
category,
Electronics,2839
Books,2809
Jewelry,2789
Toys,2768
Skincare,2762
Snacks,2759
Outdoors,2738
Apparel,2715
Sports,2706


## 12. Menyimpan Data Hasil Preprocessing

Data bersih disimpan ke `data/processed/amazon_delivery_cleaned.csv`.

Selain itu, notebook juga menyimpan ringkasan kualitas data ke `data/processed/data_quality_summary.csv` agar mudah dimasukkan ke laporan.

In [108]:
processed_file = PROCESSED_DIR / "amazon_delivery_cleaned.csv"
summary_file = PROCESSED_DIR / "data_quality_summary.csv"
metrics_file = PROCESSED_DIR / "priority_risk_delivery_time_metrics.csv"
quintile_file = PROCESSED_DIR / "priority_risk_delivery_time_quintile_summary.csv"

df.to_csv(processed_file, index=False)

quality_summary = pd.DataFrame({
    "metric": [
        "raw_rows",
        "raw_columns",
        "processed_rows",
        "processed_columns",
        "removed_rows",
        "created_at"
    ],
    "value": [
        df_raw.shape[0],
        df_raw.shape[1],
        df.shape[0],
        df.shape[1],
        df_raw.shape[0] - df.shape[0],
        time.strftime("%Y-%m-%d %H:%M:%S")
    ]
})
quality_summary.to_csv(summary_file, index=False)

if {"priority_risk_score", "delivery_time"}.issubset(df.columns):
    pearson = df["priority_risk_score"].corr(df["delivery_time"], method="pearson")
    spearman = df["priority_risk_score"].rank().corr(df["delivery_time"].rank(), method="pearson")
    n_top = int(len(df) * 0.2)
    top20 = df.nlargest(n_top, "priority_risk_score")
    bottom20 = df.nsmallest(n_top, "priority_risk_score")

    historical_metrics = pd.DataFrame({
        "metric": [
            "pearson_priority_risk_vs_delivery_time",
            "spearman_priority_risk_vs_delivery_time",
            "top20_avg_delivery_time",
            "bottom20_avg_delivery_time",
            "top20_minus_bottom20_delivery_time"
        ],
        "value": [
            pearson,
            spearman,
            top20["delivery_time"].mean(),
            bottom20["delivery_time"].mean(),
            top20["delivery_time"].mean() - bottom20["delivery_time"].mean()
        ]
    })
    historical_metrics["value"] = historical_metrics["value"].round(4)
    historical_metrics.to_csv(metrics_file, index=False)

    df_eval = df.copy()
    df_eval["risk_bin"] = pd.qcut(
        df_eval["priority_risk_score"],
        q=5,
        labels=["Q1 sangat rendah", "Q2 rendah", "Q3 sedang", "Q4 tinggi", "Q5 sangat tinggi"],
        duplicates="drop"
    )
    quintile_summary = df_eval.groupby("risk_bin", observed=True).agg(
        order_count=("order_id", "count"),
        risk_min=("priority_risk_score", "min"),
        risk_max=("priority_risk_score", "max"),
        avg_delivery_time=("delivery_time", "mean"),
        median_delivery_time=("delivery_time", "median"),
        avg_optimization_cost=("optimization_cost", "mean")
    ).reset_index()
    numeric_cols = ["risk_min", "risk_max", "avg_delivery_time", "median_delivery_time", "avg_optimization_cost"]
    quintile_summary[numeric_cols] = quintile_summary[numeric_cols].round(4)
    quintile_summary.to_csv(quintile_file, index=False)

print("Data bersih tersimpan di:", processed_file)
print("Ringkasan kualitas data tersimpan di:", summary_file)
if metrics_file.exists():
    print("Metrik evaluasi historis tersimpan di:", metrics_file)
if quintile_file.exists():
    print("Ringkasan quintile risiko tersimpan di:", quintile_file)
display(quality_summary)

Data bersih tersimpan di: c:\Users\piosg\Documents\Evolusi\data\processed\amazon_delivery_cleaned.csv
Ringkasan kualitas data tersimpan di: c:\Users\piosg\Documents\Evolusi\data\processed\data_quality_summary.csv
Metrik evaluasi historis tersimpan di: c:\Users\piosg\Documents\Evolusi\data\processed\priority_risk_delivery_time_metrics.csv
Ringkasan quintile risiko tersimpan di: c:\Users\piosg\Documents\Evolusi\data\processed\priority_risk_delivery_time_quintile_summary.csv


,metric,value
0,raw_rows,43739
1,raw_columns,16
2,processed_rows,43551
3,processed_columns,28
4,removed_rows,188
5,created_at,2026-06-15 16:32:59


## 13. Formulasi Masalah untuk Laporan

Bagian ini dapat langsung digunakan sebagai bahan laporan.

**Judul proyek yang disarankan:**

Optimasi Bobot Skor Prioritas Pengiriman Amazon Menggunakan Genetic Algorithm Berdasarkan Data Historis Last-Mile Delivery.

**Masalah optimasi:**

Proyek ini memodelkan masalah **optimasi bobot skor prioritas pengiriman** pada data last-mile delivery Amazon. Setiap baris dataset dianggap sebagai satu order/pengiriman yang berdiri sendiri. Tujuannya adalah mencari kombinasi bobot terbaik untuk membentuk `priority_score`, sehingga skor prioritas tersebut paling sesuai dengan pola `delivery_time` historis.

**Batasan dataset:**

Dataset ini bukan data rute kendaraan lengkap. Dataset hanya menyediakan informasi per order, seperti lokasi toko, lokasi tujuan, cuaca, traffic, kendaraan yang digunakan, dan waktu pengiriman historis. Dataset tidak menyediakan kapasitas kendaraan, jumlah paket dalam satu kendaraan, volume/berat paket, daftar order dalam satu trip, jumlah kendaraan tersedia, atau urutan drop-off aktual. Karena itu, kolom `vehicle` digunakan sebagai faktor kondisi pengiriman, bukan sebagai kapasitas kendaraan.

**Variabel keputusan:**

Variabel keputusan adalah bobot dari lima faktor risiko pengiriman. Satu solusi Genetic Algorithm direpresentasikan sebagai kromosom berisi lima bobot.

Contoh:

`X = [w_distance, w_traffic, w_weather, w_vehicle, w_agent]`

Contoh nilai kromosom:

`X = [0.30, 0.25, 0.15, 0.10, 0.20]`

Artinya, model memberi bobot 30% untuk jarak, 25% untuk traffic, 15% untuk cuaca, 10% untuk kendaraan, dan 20% untuk rating kurir.

**Skor prioritas berbobot:**

`priority_score_i = w_distance * distance_score_i + w_traffic * traffic_score_i + w_weather * weather_score_i + w_vehicle * vehicle_score_i + w_agent * agent_score_i`

`delivery_time` tidak dipakai sebagai input skor prioritas untuk menghindari data leakage. Pada skenario nyata, waktu pengiriman baru diketahui setelah order selesai dikirim. Dalam proyek ini, `delivery_time` hanya digunakan sebagai target evaluasi historis.

**Fungsi objektif:**

Fungsi objektif adalah memaksimalkan kesesuaian antara `priority_score` dan `delivery_time` historis.

`Maximize Spearman(priority_score, delivery_time)`

Spearman correlation dipilih karena yang ingin dicari adalah kesesuaian peringkat: order dengan risiko lebih tinggi sebaiknya memiliki skor prioritas lebih tinggi.

**Fitness function:**

Karena nilai Spearman berada pada rentang -1 sampai 1, fitness dapat dibuat menjadi rentang 0 sampai 1:

`Fitness = (Spearman(priority_score, delivery_time) + 1) / 2`

Semakin besar fitness, semakin baik kombinasi bobot yang ditemukan oleh Genetic Algorithm.

**Kendala:**

1. Setiap bobot berada pada rentang 0 sampai 1.
2. Total bobot harus sama dengan 1.
3. `delivery_time` tidak boleh menjadi input `priority_score`.
4. `delivery_time` hanya digunakan untuk evaluasi historis.
5. Setiap baris dataset dianggap sebagai satu order/pengiriman independen.
6. Pesanan yang digunakan harus memiliki koordinat toko dan tujuan yang valid.
7. Jarak pengiriman harus realistis untuk konteks last-mile delivery, yaitu tidak lebih dari 100 km.
8. Kolom `vehicle` hanya digunakan sebagai faktor penalti, bukan sebagai kapasitas kendaraan.
9. Model tidak memasukkan kapasitas kendaraan, berat/volume paket, jumlah kendaraan, atau multi-drop route karena informasi tersebut tidak tersedia dalam dataset.

**Output model:**

Output utama dari Genetic Algorithm adalah kombinasi bobot terbaik. Bobot tersebut kemudian digunakan untuk menghitung `priority_score` dan mengurutkan order dari prioritas tertinggi ke terendah.